In [1]:
%load_ext autoreload
%autoreload 2

# Compile experiment results into a dictionary

In [3]:
import os
import pickle

from tabpfn_project.paths import CONTEXT_SIZES_DIR

metadata_dir = CONTEXT_SIZES_DIR / "metadata"
tabpfn_preds_dir = CONTEXT_SIZES_DIR / "tabpfn_preds_full"


In [ ]:
def fetch_save_dict(metadata_dir):

    experiment_results_dict = {}

    for fname in os.listdir(metadata_dir):
        if not fname.endswith('.pkl'):
            continue
        fpath = os.path.join(metadata_dir, fname)
        with open(fpath, 'rb') as f:
            results_dict = pickle.load(f)

        model_name   = results_dict['model_name']
        scenario     = results_dict['scenario']
        context_size = results_dict['context_size']
        fold         = results_dict['fold']
        seed_context = results_dict['seed_context']

        experiment_results_dict \
            .setdefault(model_name, {}) \
            .setdefault(scenario, {}) \
            .setdefault(context_size, {}) \
            .setdefault(fold, {})[seed_context] = results_dict



    save_path = os.path.join(metadata_dir, "experiment_results_dict.pkl")
    with open(save_path, 'wb') as f:
        pickle.dump(experiment_results_dict, f)
    print(f"Saved to {save_path}")

In [ ]:
load_path = os.path.join(metadata_dir, "experiment_results_dict.pkl")
with open(load_path, 'rb') as f:
    experiment_results_dict = pickle.load(f)
print("Loaded successfully.")
print("Models:", list(experiment_results_dict.keys()))

# Helpers

In [ ]:
import numpy as np
from tabpfn_project.helper.distnet_helpers import data_source_release, load_data
from sklearn.model_selection import KFold

def load_y_test(scenario_name, fold_idx):
    sc_dict = data_source_release.get_sc_dict()
    data_dir = data_source_release.get_data_dir()

    runtimes, _, _ = load_data.get_data(
        scenario=scenario_name, 
        data_dir=data_dir,
        sc_dict=sc_dict,
        retrieve=sc_dict[scenario_name]['use']
    )
        
    runtimes = np.asarray(runtimes)

    # Get CV splits
    kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
    splits = list(kf.split(np.arange(runtimes.shape[0])))
    _, test_idx = splits[fold_idx]  # process the specified fold

    #------------------------------------#
    y_test = runtimes[test_idx]
    return y_test

def load_tabpfn_preds(scenario_name, fold_idx, context_size, context_seed, seed_features, seed_samples_per_instance, feature_drop_rate, target_scale, subsample_method, num_samples_per_instance, use_cpu):
    fname = (
        f"tabpfn_{scenario_name}_{fold_idx}_{context_seed}_{seed_features}_{seed_samples_per_instance}_{feature_drop_rate}_"
                         f"{context_size}_{target_scale}_{subsample_method}_{num_samples_per_instance}_{'cpu' if use_cpu else 'gpu'}_test_preds.pkl"
    )
    fpath = os.path.join(tabpfn_preds_dir, fname)
    with open(fpath, 'rb') as f:
        preds = pickle.load(f)
    return preds
    

# Calculate metrics of interest and store results

In [ ]:
from tabpfn_project.helper import (
    calculate_all_distribution_metrics_distnet_logspace,
)
from tabpfn_project.helper.pfn_helpers import calculate_all_distribution_metrics_tabpfn_logspace


def calculate_all_metrics():
    tabpfn_context_size_results = []
    distnet_context_size_results = []

    total_experiments = len(SCENARIOS) * len(MODELS) * len(CONTEXT_SIZES) * len(FOLDS) * len(CONTEXT_SEEDS)
    current_experiment = 0
    for scenario in SCENARIOS:
        for model_name in MODELS:
            for context_size in CONTEXT_SIZES:
                for fold in FOLDS:
                    y_test = load_y_test(scenario, fold)
                    for context_seed in CONTEXT_SEEDS:
                        case_dict = experiment_results_dict[model_name][scenario][context_size][fold][context_seed]

                        if model_name == 'tabpfn':
                            tabpfn_preds = load_tabpfn_preds(scenario, fold, context_size, context_seed, case_dict['seed_features'],
                                                            case_dict['seed_samples_per_instance'], case_dict['feature_drop_rate'],
                                                            case_dict['target_scale'], case_dict['subsample_method'],
                                                            case_dict['num_samples_per_instance'], case_dict['use_cpu'])
                            metrics_summary, instance_summary = calculate_all_distribution_metrics_tabpfn_logspace(y_test, tabpfn_preds, case_dict['target_scale'], N_GRID_POINTS)
                            tabpfn_context_size_results.append({
                                "scenario": scenario, "model": model_name, "context_size": context_size,
                                "fold": fold, "context_seed": context_seed,
                                "metrics_summ0ary": metrics_summary, "instance_summary": instance_summary
                            })

                        elif model_name == 'distnet':
                            metrics_summary, instance_summary = calculate_all_distribution_metrics_distnet_logspace(y_test, case_dict['test_preds'], case_dict['y_scale'], N_GRID_POINTS)
                            distnet_context_size_results.append({
                                "scenario": scenario, "model": model_name, "context_size": context_size,
                                "fold": fold, "context_seed": context_seed,
                                "metrics_summary": metrics_summary, "instance_summary": instance_summary
                            })
                        
                        current_experiment += 1
                        print(f"✅Completed experiment {current_experiment}/{total_experiments}")
                        results_dir = r"C:\Users\ihagv\Desktop\Study Project UFR\Study_Project_Code\experiments\tabpfn_vs_distnet\results"
                        os.makedirs(results_dir, exist_ok=True)

                        tabpfn_save_path = os.path.join(results_dir, "tabpfn_context_size_results.pkl")
                        with open(tabpfn_save_path, 'wb') as f:
                            pickle.dump(tabpfn_context_size_results, f)
                        print(f"Saved TabPFN results ({len(tabpfn_context_size_results)} entries) to {tabpfn_save_path}")

                        distnet_save_path = os.path.join(results_dir, "distnet_context_size_results.pkl")
                        with open(distnet_save_path, 'wb') as f:
                            pickle.dump(distnet_context_size_results, f)
                        print(f"Saved DistNet results ({len(distnet_context_size_results)} entries) to {distnet_save_path}")

### Visualization

In [ ]:

import pickle
from collections import defaultdict

import numpy as np

results_dir = r"C:\Users\ihagv\Desktop\Study Project UFR\Study_Project_Code\experiments\tabpfn_vs_distnet\results"

with open(f"{results_dir}/tabpfn_context_size_results.pkl", "rb") as f:
    tabpfn_results = pickle.load(f)

with open(f"{results_dir}/distnet_context_size_results.pkl", "rb") as f:
    distnet_results = pickle.load(f)

all_results = tabpfn_results + distnet_results

METRICS = ["NLLH", "CRPS", "Wasserstein", "KS"]

# Aggregate: for each (model, scenario, context_size) concatenate all per-instance arrays
# across all folds and seeds, then compute mean and std.
agg = defaultdict(lambda: defaultdict(list))
# agg[(model, scenario, context_size)][metric] -> list of per-instance values

for entry in all_results:
    model        = entry["model"]
    scenario     = entry["scenario"]
    context_size = entry["context_size"]
    inst_summ    = entry["instance_summary"]
    key = (model, scenario, context_size)
    for metric in METRICS:
        vals = inst_summ[metric]
        # handle both torch tensors and numpy arrays
        if hasattr(vals, "cpu"):
            vals = vals.cpu().numpy()
        vals = np.asarray(vals).ravel()
        agg[key][metric].append(vals)

# Compute final mean / std per (model, scenario, context_size, metric)
plot_data = {}
for key, metric_dict in agg.items():
    plot_data[key] = {}
    for metric, arrays in metric_dict.items():
        combined = np.concatenate(arrays)
        plot_data[key][metric] = (combined.mean(), combined.std(), len(combined))

# Collect unique scenarios and context_sizes (sorted)
scenarios     = sorted({key[1] for key in plot_data})
context_sizes = sorted({key[2] for key in plot_data})
models        = ["tabpfn", "distnet"]

print(f"Scenarios: {scenarios}")
print(f"Context sizes: {context_sizes}")
print(f"Models: {models}")
print(f"Total aggregated entries: {len(plot_data)}")


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

MODEL_COLORS  = {"tabpfn": "#1f77b4", "distnet": "#d62728"}
MODEL_LABELS  = {"tabpfn": "TabPFN",  "distnet": "DistNet"}
METRIC_LABELS = {"NLLH": "NLLH", "CRPS": "CRPS", "Wasserstein": "Wasserstein", "KS": "KS"}

n_rows = len(scenarios)
n_cols = len(METRICS)

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(5 * n_cols, 4 * n_rows),
    squeeze=False,
)

for row_idx, scenario in enumerate(scenarios):
    for col_idx, metric in enumerate(METRICS):
        ax = axes[row_idx][col_idx]

        for model in models:
            xs, means, stds = [], [], []
            for cs in context_sizes:
                key = (model, scenario, cs)
                if key in plot_data and metric in plot_data[key]:
                    m, s, n = plot_data[key][metric]
                    xs.append(cs)
                    means.append(m)
                    stds.append(s / np.sqrt(n))

            if not xs:
                continue

            xs    = np.array(xs,    dtype=float)
            means = np.array(means, dtype=float)
            stds  = np.array(stds,  dtype=float)

            color = MODEL_COLORS[model]
            ax.plot(xs, means, marker="o", markersize=4, linewidth=1.8,
                    color=color, label=MODEL_LABELS[model])
            ax.fill_between(xs, means - stds, means + stds,
                            alpha=0.20, color=color)

        ax.set_xscale("log", base=2)
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
        ax.xaxis.set_minor_formatter(ticker.NullFormatter())
        ax.set_xticks(context_sizes)
        ax.tick_params(axis="x", labelrotation=45, labelsize=8)

        if metric in ("CRPS", "Wasserstein", "KS"):
            ax.set_yscale("log")

        ax.set_xlabel("Context Size", fontsize=9)
        ax.set_ylabel(METRIC_LABELS[metric], fontsize=9)

        if row_idx == 0:
            ax.set_title(METRIC_LABELS[metric], fontsize=11, fontweight="bold", pad=8)

        if col_idx == 0:
            ax.set_ylabel(
                f"{scenario}\n\n{METRIC_LABELS[metric]}",
                fontsize=9,
            )

        ax.grid(True, which="major", linestyle="--", linewidth=0.5, alpha=0.6)

# Shared legend
handles = [
    matplotlib.lines.Line2D([0], [0], color=MODEL_COLORS[m], linewidth=2, label=MODEL_LABELS[m])
    for m in models
]
fig.legend(handles=handles, loc="upper right", fontsize=11, framealpha=0.9,
           bbox_to_anchor=(1.0, 1.0))

plt.suptitle(
    "TabPFN vs DistNet — Metric quality across context sizes (shaded = ±1 SE)",
    fontsize=14, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.show()


In [ ]:
import os

import numpy as np
import pandas as pd

rows = []
for scenario in scenarios:
    for cs in context_sizes:
        row = {"Scenario": scenario, "Context Size": cs}
        for model in models:
            key = (model, scenario, cs)
            if key in plot_data:
                for metric in METRICS:
                    m, s, n = plot_data[key][metric]
                    row[f"{MODEL_LABELS[model]} {metric} mean"] = round(float(m), 4)
                    row[f"{MODEL_LABELS[model]} {metric} SE"]   = round(float(s / np.sqrt(n)), 4)
            else:
                for metric in METRICS:
                    row[f"{MODEL_LABELS[model]} {metric} mean"] = None
                    row[f"{MODEL_LABELS[model]} {metric} SE"]   = None
        rows.append(row)

df = pd.DataFrame(rows)

# Build a multi-level column index: (metric, model stat) for display
col_order = ["Scenario", "Context Size"]
for metric in METRICS:
    for model in models:
        col_order.append(f"{MODEL_LABELS[model]} {metric} mean")
        col_order.append(f"{MODEL_LABELS[model]} {metric} SE")
df = df[col_order]

# Pretty display with multi-level header
metric_cols = {}
for metric in METRICS:
    for model in models:
        metric_cols[f"{MODEL_LABELS[model]} {metric} mean"] = (metric, MODEL_LABELS[model], "mean")
        metric_cols[f"{MODEL_LABELS[model]} {metric} SE"]   = (metric, MODEL_LABELS[model], "SE")

tuples = [("", "", col) if col in ("Scenario", "Context Size")
          else metric_cols[col]
          for col in df.columns]
df.columns = pd.MultiIndex.from_tuples(tuples)

pd.set_option("display.max_rows",    None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width",       None)
pd.set_option("display.float_format", "{:.4f}".format)

# Save to CSV (multi-level header flattened to "Metric_Model_stat")
results_dir = r"C:\Users\ihagv\Desktop\Study Project UFR\Study_Project_Code\experiments\tabpfn_vs_distnet\results"
csv_path = os.path.join(results_dir, "context_size_metrics_table.csv")
df_csv = df.copy()
df_csv.columns = [
    "_".join(filter(None, col)).strip("_") if isinstance(col, tuple) else col
    for col in df_csv.columns
]
df_csv.to_csv(csv_path, index=False)
print(f"Table saved to {csv_path}")

df
